# M22 · LLM-as-judge validation

_AFP-AI · Domain 4 · GenAI_

We simulate a judge and humans, then compute agreement, correlation, calibration, and position bias. The key statistic is Cohen's $\kappa=(p_o-p_e)/(1-p_e)$.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import cohen_kappa_score
from sklearn.metrics import confusion_matrix

rng = np.random.default_rng(22)

## Simulate human labels and judge labels

Labels are binary for simplicity: 1 means the creative response passes the rubric, 0 means it does not.

In [ ]:
n = 240
human = rng.binomial(1, 0.58, size=n)
flip = rng.binomial(1, 0.18, size=n)
judge = np.where(flip == 1, 1 - human, human)

print(np.bincount(human))
print(np.bincount(judge))
assert len(human) == len(judge)

## Cohen's kappa

Raw agreement can look good when both raters overuse the same label. Kappa subtracts chance agreement implied by label frequencies.

In [ ]:
observed = np.mean(human == judge)
human_pos = human.mean()
judge_pos = judge.mean()
expected = human_pos * judge_pos + (1.0 - human_pos) * (1.0 - judge_pos)
kappa_manual = (observed - expected) / (1.0 - expected)
kappa_sklearn = cohen_kappa_score(human, judge)

print(round(observed, 3))
print(round(expected, 3))
print(round(kappa_manual, 3))
assert np.isclose(kappa_manual, kappa_sklearn)

In [ ]:
cm = confusion_matrix(human, judge)
cm_df = pd.DataFrame(cm, index=["human_fail", "human_pass"], columns=["judge_fail", "judge_pass"])
print(cm_df)

## Numeric judge scores

For score rubrics, correlation asks whether the judge moves with humans. Calibration asks whether a score value means what it claims.

In [ ]:
human_score = rng.normal(loc=3.2 + 1.2 * human, scale=0.55, size=n)
judge_score = 0.7 * human_score + rng.normal(loc=0.8, scale=0.45, size=n)
human_score = np.clip(human_score, 1.0, 5.0)
judge_score = np.clip(judge_score, 1.0, 5.0)
correlation = np.corrcoef(human_score, judge_score)[0, 1]

print(round(correlation, 3))
assert correlation > 0.5

In [ ]:
bins = pd.cut(judge_score, bins=[1, 2, 3, 4, 5], include_lowest=True)
calibration = pd.DataFrame({"human_pass": human, "judge_score": judge_score, "bin": bins})
calibration_table = calibration.groupby("bin", observed=False).agg(
    items=("human_pass", "size"),
    human_pass_rate=("human_pass", "mean"),
    avg_judge_score=("judge_score", "mean")
)

print(calibration_table.round(3))

## Position-bias demo

A fair pairwise judge should not prefer whichever answer is shown first. We simulate a judge with a first-position boost, then repeat with swapped order.

In [ ]:
pairs = 300
true_quality_a = rng.normal(size=pairs)
true_quality_b = rng.normal(size=pairs)
position_boost = 0.35
noise_ab = rng.normal(scale=0.25, size=pairs)
noise_ba = rng.normal(scale=0.25, size=pairs)
score_a_first = true_quality_a + position_boost + noise_ab
score_b_second = true_quality_b
score_b_first = true_quality_b + position_boost + noise_ba
score_a_second = true_quality_a
choose_a_when_first = score_a_first > score_b_second
choose_b_when_first = score_b_first > score_a_second
first_position_wins = np.mean(np.r_[choose_a_when_first, choose_b_when_first])

print(round(first_position_wins, 3))
assert first_position_wins > 0.55

In [ ]:
stable_a_win = np.mean(choose_a_when_first & np.logical_not(choose_b_when_first))
stable_b_win = np.mean(np.logical_not(choose_a_when_first) & choose_b_when_first)
flip_rate = np.mean(choose_a_when_first == choose_b_when_first)

print("stable A win", round(stable_a_win, 3))
print("stable B win", round(stable_b_win, 3))
print("position-driven flip", round(flip_rate, 3))

## Visual checks

Plots make judge behavior easier to explain to stakeholders: confusion matrices for labels, scatterplots for score agreement, and bar charts for bias probes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].scatter(human_score, judge_score, alpha=0.5, s=15)
axes[0].set_xlabel("human score")
axes[0].set_ylabel("judge score")
axes[0].set_title("score agreement")
axes[1].bar(["first position", "neutral"], [first_position_wins, 0.5], color=["#f58518", "#4c78a8"])
axes[1].set_ylim(0, 1)
axes[1].set_title("position bias")
plt.show()

## Takeaway

A judge can accelerate evaluation only after it is measured. Always report agreement, calibration, and bias probes by the slices that matter for the product.